In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "06-gateway/scaling-admission-cost/agentic-scaling-lab-mistral/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 04 · From a load test to Kubernetes and vLLM settings

**What you'll learn.** How the overload feedback loop looks in a load test — on the API and on a fleet — what
admission control and spill-over do to it, and how to turn measurements into settings: orchestrator pods, vLLM
replicas, `--max-num-seqs`, the queue cap, the in-flight cap, KEDA thresholds — and into the "what breaks first"
answer. Code: `scalelab/sim.py`.

> **In a design review.** *"Before the customer commits GPUs I run a load test with the incident intent mix on one replica:
> it gives me tokens per second at the batch that meets the latency target, which sizes the fleet; the turn
> duration, which with Little's law gives in-flight turns and the cap; and the point where spilling to the API or
> shedding is kinder than queueing."*

In [ ]:
import sys, asyncio, inspect, random
sys.path[:0] = [".", ".."]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from nbutil import todo, check, acheck, latency_cdfs, timeline
from scalelab.clock import CLOCK
%matplotlib inline
pd.set_option("display.width", 160)
CLOCK.reset(0.02)   # 50x faster than real time

## 1. Nine regimes

The same 120 users against (a) a simulated 3 M TPM API limit and (b) two Ministral 14B replicas on H100s:
naive (retries only), degrade levels without a cap, degrade levels with the cap the capacity model derives, a
tighter cap, and the hybrid (fleet first, API when the fleet is saturated). Each run is 90 virtual seconds.

In [ ]:
from scalelab.sim import make_setup, simulate, compare
results = {}
for name, mode, kw, users in [("API, moderate (20 users)", "hosted", dict(pool_tpm=3_000_000), 20),
                              ("API, overload, naive", "hosted", dict(pool_tpm=3_000_000, naive=True), 120),
                              ("API, overload, degrade only", "hosted", dict(pool_tpm=3_000_000, max_inflight=1000), 120),
                              ("API, overload, degrade + cap", "hosted", dict(pool_tpm=3_000_000), 120),
                              ("fleet, moderate (20 users)", "local", dict(replicas=2), 20),
                              ("fleet, overload, naive", "local", dict(replicas=2, naive=True), 120),
                              ("fleet, overload, degrade + cap", "local", dict(replicas=2), 120),
                              ("fleet, overload, cap 30", "local", dict(replicas=2, max_inflight=30), 120),
                              ("hybrid, overload", "hybrid", dict(replicas=2, pool_tpm=3_000_000), 120)]:
    CLOCK.reset(0.02)
    results[name] = await simulate(make_setup(mode, **kw), users=users, duration_s=90, think_s=5)
summary = compare(results).round(4)
summary[["turns", "completed", "shed_rate", "p50_s", "p95_s", "pushback_calls", "failed", "degraded_share", "max_batch", "api_share_of_calls", "cost_per_turn_usd"]]

In [ ]:
latency_cdfs({k: v for k, v in results.items() if k.startswith("API")}, "Turn latency — the API")

In [ ]:
latency_cdfs({k: v for k, v in results.items() if not k.startswith("API")}, "Turn latency — the fleet and the hybrid")

In [ ]:
timeline(results["API, overload, naive"], "API, naive: the pool saturates, retries pile up, latency explodes")

In [ ]:
timeline(results["fleet, overload, naive"], "fleet, naive: no errors, no 429s — the batch climbs and everyone slows down")

In [ ]:
timeline(results["hybrid, overload"], "hybrid: the batch holds near the target; the excess goes to the API")

Read the table with the primer's vocabulary. On the API the naive run *is* the feedback loop (429s → retries →
long turns → more in flight); degrade levels convert it into cheaper calls, and the cap converts the rest into shed
requests with a `Retry-After` while every admitted turn stays fast. On the fleet the naive run shows the loop's
second form: zero errors, zero pushback, a batch of 60 and a p95 two and a half times the moderate run — the
dashboard says healthy while every customer waits. The cap and the saturation signal bound the batch; the hybrid
keeps the batch at the target and pays the API for the rest — the only regime that serves almost everyone within
the target, and the `cost_per_turn` column is what that costs.

## 2. From measurements to settings

Take the per-replica throughput at the target batch and the turn duration from the moderate runs, apply Little's
law at the planned peak, add headroom, divide by what one pod or one replica can hold.

In [ ]:
from scalelab.capacity import Scenario, plan
s, p = Scenario(), plan(Scenario())
f = p["self_hosted"]
api_turn = results["API, moderate (20 users)"].summary()["p95_s"]
fleet_turn = results["fleet, moderate (20 users)"].summary()["p95_s"]
print(f"measured p95 turn: API {api_turn:.1f} s, fleet {fleet_turn:.1f} s (the plan assumed {s.turn_seconds:g} s and {f['turn_seconds']:.1f} s)")
for level in ("average", "peak", "incident"):
    turns_s, calls_s = p["rates"][level]["turns_per_s"], p["rates"][level]["calls_per_s"]
    pods = np.ceil(turns_s * fleet_turn * 1.4 / 80)
    by_tokens = calls_s * s.output_tokens / f["tok_s_per_replica"]          # decode throughput
    by_seqs = calls_s * f["call_seconds"] / f["batch"]                       # Little: calls in flight ÷ batch per replica
    replicas = np.ceil(max(by_tokens, by_seqs) * 1.3)
    print(f"{level:9s} {turns_s:5.1f} turns/s → {turns_s * fleet_turn:5.0f} in flight → {pods:2.0f} orchestrator pods (conc 80, ×1.4); "
          f"{calls_s:5.1f} calls/s → {by_tokens:4.1f} replicas by tokens/s, {by_seqs:4.1f} by batch → {replicas:2.0f} vLLM replicas (×1.3)")

| Setting | Value (Ministral 14B on H100) | Why |
|---|---|---|
| vLLM replicas | 11 at peak (static), KEDA to 14 for headroom | `tok/s per replica` at the target batch; cold start 3–8 min, so the peak is provisioned, not autoscaled |
| `--max-num-seqs` | 48 (2× the target batch of 24) | room above the target for bursts; beyond it TPOT is being traded away |
| `--max-num-queued-reqs` | ≈ 12 per replica | turn a queue into a 503 the gateway can act on; without it vLLM queues forever |
| `--max-model-len` | 16,384 | frees KV memory versus the 256k default; the turn never needs more |
| `--kv-cache-dtype fp8` | on | halves KV bytes per token → twice the resident sequences |
| in-flight cap (gateway) | 293 turns for the peak fleet; 175 on a 20 M TPM API limit | Little's law from the fleet's or the limit's capacity; tune from the load test |
| KEDA (orchestrator) | ready messages per pod ≈ 20 | queue depth is the honest signal; pods hold ≤ 80 turns |
| KEDA (fleet) | `vllm:num_requests_waiting` > 5 or `vllm:kv_cache_usage_perc` > 0.8 | the inference gateway's saturation thresholds |
| terminationGracePeriodSeconds | 60 (orchestrator), 120 (vLLM) | finish or checkpoint the turn; drain the batch |

Kubernetes, Postgres, Redis and RabbitMQ together are about 1 % of the model bill; the fleet or the API is the bill.

## 3. What breaks first

In [ ]:
pd.DataFrame(p["breaks_first"]).round(2)

## Your turn

Fill in each function (replace the `todo()` call), then run the check cell below it. The practice notebook runs end to end with the exercises untouched; the solutions notebook has every check passing.

#### (a) Replicas needed

Two constraints, take the larger: decode throughput (calls/s × output tokens ÷ tokens/s per replica) and resident calls (calls/s × call seconds ÷ batch per replica, Little's law); then × headroom, ceil, never below `minimum`.

In [ ]:
def replicas_needed(calls_per_s, output_tokens, tok_s_per_replica, call_seconds, batch, headroom=1.3, minimum=2):
    return todo()

In [ ]:
def _a():
    args = (s.output_tokens, f["tok_s_per_replica"], f["call_seconds"], f["batch"])
    for level in ("average", "peak", "incident"):
        assert replicas_needed(p["rates"][level]["calls_per_s"], *args) == f["replicas"][level], level
    assert replicas_needed(1, *args) == 2
check("a: replicas", _a)

#### (b) Cap from the fleet

How many turns can be in flight before the fleet passes its target batch: replicas × batch × turn seconds ÷ (calls per turn × call seconds).

In [ ]:
def cap_from_fleet(replicas, batch, turn_seconds, calls_per_turn, call_seconds):
    return todo()

In [ ]:
def _b():
    assert abs(cap_from_fleet(f["replicas"]["peak"], f["batch"], f["turn_seconds"], s.calls_per_turn, f["call_seconds"]) - f["max_inflight_turns_peak_fleet"]) < 0.01
    assert 50 < cap_from_fleet(2, 24, 10.0, 2.2, 4.1) < 56
check("b: cap from the fleet", _b)

#### (c) Fleet cost and the price of residency

Monthly dollars for `gpus` GPUs at an hourly price (730 h), and the monthly premium over the hosted planning mix.

In [ ]:
def fleet_monthly_usd(gpus, usd_per_gpu_hour):
    return todo()

def residency_premium_usd(gpus, usd_per_gpu_hour, hosted_monthly_usd):
    return todo()

In [ ]:
def _c():
    hosted = p["hosted"]["monthly_usd"][p["hosted"]["planning_mix"]]
    assert abs(fleet_monthly_usd(11, 6.88) - f["monthly_usd"]["peak"]) < 1
    assert residency_premium_usd(11, 6.88, hosted) > 10_000 and residency_premium_usd(11, 3.75, hosted) < 0
    print(f"   keeping data in Singapore costs ${residency_premium_usd(11, 6.88, hosted):,.0f}/month more on on-demand H100s and "
          f"${-residency_premium_usd(11, 4.86, hosted):,.0f}/month less on 3-year committed ones")
check("c: fleet cost", _c)

#### (d) Choose the cap

Pick an in-flight cap for two replicas with 120 users so that p95 ≤ 7.5 s and no turn fails; keep shedding under 45 %. The check runs a simulation with your value.

In [ ]:
MY_CAP = None   # an integer

In [ ]:
async def _d():
    if MY_CAP is None:
        todo()
    CLOCK.reset(0.02)
    r = await simulate(make_setup("local", replicas=2, max_inflight=MY_CAP), users=120, duration_s=60, think_s=5)
    s_ = r.summary()
    print(f"   cap {MY_CAP}: p95 {s_['p95_s']:.1f} s, shed {s_['shed_rate']:.0%}, failed {s_['failed']}, max batch {s_['max_batch']}")
    assert s_["p95_s"] <= 7.5 and s_["failed"] == 0 and s_["shed_rate"] < 0.45
await acheck("d: choose the cap", _d)

## Takeaways for the conversation

- A load test gives you tokens/s per replica at the target batch and the turn duration; Little's law gives in-flight turns; headroom and per-replica or per-pod capacity give the counts.
- The naive fleet does not fail — it slows down for everyone with a green dashboard. Cap the queue, watch the batch, and let admission control convert overload into bounded latency plus honest `Retry-After`s.
- The hybrid is the regime that serves almost everyone within the target; its price is the API share, and its constraint is the data policy.
- Say what breaks first: the API limit or the peak-sized fleet at the incident, then the billing system; Kubernetes is not on the list.

## Verify before the conversation

vLLM 0.29 flags and defaults (`max_num_seqs` 128, `max_num_batched_tokens` 16,384, `--max-num-queued-reqs`); KEDA scaler syntax for Prometheus metrics; GPU node-pool cold-start times in the customer's cloud; whether the Gateway API Inference Extension's flow control has left alpha.